# 01 - Dataset Overview and City Selection

This notebook documents the first project decision: choosing a manageable geographic scope from the Yelp Open Dataset.

The full Yelp dataset is large, so the professor's recommendation to focus on one city or metropolitan area is important. We begin with the business table because it is much smaller than the review and user files, and it contains the city/state fields needed for scope selection.

## Goals

- Confirm the raw Yelp files are available.
- Count businesses by city and state.
- Select a city that is large enough for meaningful analysis but manageable for academic work.
- Save the city-count table for documentation.

In [1]:
from pathlib import Path
import csv
import json
from collections import Counter
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_YELP_DIR = DATA_DIR / "raw" / "yelp"
INTERIM_DIR = DATA_DIR / "interim"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

BUSINESS_PATH = RAW_YELP_DIR / "yelp_academic_dataset_business.json"
REVIEW_PATH = RAW_YELP_DIR / "yelp_academic_dataset_review.json"
USER_PATH = RAW_YELP_DIR / "yelp_academic_dataset_user.json"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp


In [2]:
expected_files = [
    "yelp_academic_dataset_business.json",
    "yelp_academic_dataset_review.json",
    "yelp_academic_dataset_user.json",
    "yelp_academic_dataset_checkin.json",
    "yelp_academic_dataset_tip.json",
]

for filename in expected_files:
    path = RAW_YELP_DIR / filename
    size_mb = path.stat().st_size / (1024 * 1024) if path.exists() else 0
    print(f"{filename:<40} exists={path.exists():<5} size_mb={size_mb:,.1f}")

yelp_academic_dataset_business.json      exists=1     size_mb=113.4
yelp_academic_dataset_review.json        exists=1     size_mb=5,094.4
yelp_academic_dataset_user.json          exists=1     size_mb=3,207.5
yelp_academic_dataset_checkin.json       exists=1     size_mb=273.7
yelp_academic_dataset_tip.json           exists=1     size_mb=172.2


In [3]:
def iter_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                yield json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} at line {line_number}") from exc

city_state_counts = Counter()

for record in iter_jsonl(BUSINESS_PATH):
    city = (record.get("city") or "").strip()
    state = (record.get("state") or "").strip()
    if city:
        city_state_counts[(city, state)] += 1

city_rows = [
    {"city": city, "state": state, "business_count": count}
    for (city, state), count in city_state_counts.items()
]
city_rows = sorted(city_rows, key=lambda row: (-row["business_count"], row["city"], row["state"]))

city_counts_path = OUTPUTS_DIR / "city_business_counts.csv"
with city_counts_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["city", "state", "business_count"])
    writer.writeheader()
    writer.writerows(city_rows)

print(f"Saved: {city_counts_path}")
print("\nTop 20 city/state combinations:")
for row in city_rows[:20]:
    print(f"{row['city']:<24} {row['state']:<4} {row['business_count']:>8}")

Saved: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\outputs\city_business_counts.csv

Top 20 city/state combinations:
Philadelphia             PA      14568
Tucson                   AZ       9251
Tampa                    FL       9049
Indianapolis             IN       7543
Nashville                TN       6971
New Orleans              LA       6208
Reno                     NV       5934
Edmonton                 AB       5054
Saint Louis              MO       4828
Santa Barbara            CA       3834
Boise                    ID       2938
Clearwater               FL       2221
Saint Petersburg         FL       1663
Metairie                 LA       1644
Sparks                   NV       1624
Wilmington               DE       1447
Franklin                 TN       1411
St. Louis                MO       1254
St. Petersburg           FL       1185
Meridian                 ID       1043


## City Choice

We select **New Orleans, Louisiana** for the academic phase.

Reasoning:

- It is large enough to support meaningful review forecasting and social network analysis.
- It is smaller than the largest cities such as Philadelphia, Tucson, and Tampa, making iteration faster.
- It has a clear city/state identity, which simplifies explanation and filtering.

## City Normalization Note

The initial city-count inspection and the final city extraction may produce slightly different counts. The inspection table is used for orientation, while the extraction step applies cleaner matching:

- city names are stripped of leading/trailing whitespace,
- city matching is case-insensitive,
- state abbreviations are stripped and uppercased.

For New Orleans, the first inspection showed **6,208** businesses, while the normalized extraction selected **6,215** businesses. The normalized count is used for the project because it better handles small formatting inconsistencies in the raw data.